# 🎙️ Speech Emotion Recognition - Exploratory Analysis

This notebook demonstrates how to use the Speech Emotion Recognition system.

## 1. Setup and Imports

In [ ]:
import sys
import os

# Add src to path
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

from src import config
from src.preprocessing import preprocess_audio, load_audio
from src.feature_extraction import extract_all_features, extract_mel_spectrogram_for_cnn
from src.dataset_loader import load_all_datasets
from src.utils import plot_waveform, plot_spectrogram

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All imports successful!")

## 2. Load and Explore Dataset

In [ ]:
# Load datasets
df = load_all_datasets()
df.head()

In [ ]:
# Dataset statistics
print("Dataset Shape:", df.shape)
print("\nDataset Info:")
print(df.info())
print("\nEmotion Distribution:")
print(df['emotion'].value_counts())

In [ ]:
# Visualize emotion distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Emotion counts
df['emotion'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Emotion Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Dataset distribution
df['dataset'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Dataset Distribution', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 3. Audio Preprocessing and Visualization

In [ ]:
# Load a sample audio file
sample_file = df.iloc[0]['file_path']
sample_emotion = df.iloc[0]['emotion']

print(f"Sample File: {sample_file}")
print(f"Emotion: {sample_emotion}")

# Load audio
audio, sr = load_audio(sample_file)
print(f"\nAudio Shape: {audio.shape}")
print(f"Sample Rate: {sr} Hz")
print(f"Duration: {len(audio)/sr:.2f} seconds")

In [ ]:
# Plot waveform
plt_wave = plot_waveform(audio, sr, title=f'Waveform - {sample_emotion.upper()}')
plt.show()

In [ ]:
# Plot spectrogram
mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

plt_spec = plot_spectrogram(mel_spec_db, sr, 512, title=f'Mel Spectrogram - {sample_emotion.upper()}')
plt.show()

## 4. Feature Extraction

In [ ]:
# Extract features
features = extract_all_features(audio, sr)
print(f"Extracted Features Shape: {features.shape}")
print(f"Number of Features: {len(features)}")

In [ ]:
# Extract mel-spectrogram for CNN
mel_spec_cnn = extract_mel_spectrogram_for_cnn(audio, sr)
print(f"Mel-Spectrogram Shape for CNN: {mel_spec_cnn.shape}")

# Visualize
plt.figure(figsize=(10, 6))
librosa.display.specshow(mel_spec_cnn, sr=sr, x_axis='time', y_axis='mel', cmap='viridis')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel-Spectrogram for CNN Input')
plt.tight_layout()
plt.show()

## 5. Compare Different Emotions

In [ ]:
# Select one sample from each emotion
emotion_samples = df.groupby('emotion').first().reset_index()

fig, axes = plt.subplots(len(emotion_samples), 1, figsize=(15, 3*len(emotion_samples)))

for idx, row in emotion_samples.iterrows():
    audio, sr = load_audio(row['file_path'], duration=3)
    
    axes[idx].plot(np.linspace(0, len(audio)/sr, len(audio)), audio)
    axes[idx].set_title(f"Emotion: {row['emotion'].upper()}", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Time (s)')
    axes[idx].set_ylabel('Amplitude')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Extract Features for All Samples (Optional)

In [ ]:
# This cell extracts features for all samples - may take a while!
# Uncomment to run

# from src.feature_extraction import batch_extract_features
# from src.utils import save_features

# # Extract features
# features, labels = batch_extract_features(
#     df['file_path'].tolist(),
#     df['label'].tolist(),
#     feature_type='mel_spectrogram'
# )

# # Save features
# save_features(features, labels)

## 7. Next Steps

After exploring the data:
1. Extract features for all samples using the cell above
2. Train a model using `python src/train.py`
3. Make predictions using `python src/predict.py --audio_path <path>`
4. Launch the web app using `streamlit run app.py`